# U06 應用開發（二）：完整應用模式 ＋ 內幕（一）儲存引擎

**資料庫管理**・統計系三年級・10/15　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/notebooks/unit06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

上半場把你的 App 長成**完整 CRUD**、學會**擋住併發競態**（共同要求核心）；下半場掀開引擎蓋：**資料到底怎麼躺在磁碟上**——教師現場實作一顆 mini pager。

> **投影片式 notebook 使用法**：上課跟著往下走，程式格按 `Shift+Enter` 執行；左側「目錄」可跳節。回家可以重跑、改參數做實驗——**講義是可以跑的**。
>
> 開始前建議：檔案 → 在雲端硬碟中儲存副本，改動才會留下來。

## 0. 本單元地圖（135 分鐘）

| 節 | 分鐘 | 內容 | 與專題的關係 |
|---|---|---|---|
| 第 1 節（應用） | 50 | 完整 CRUD 模板（編輯／刪除／下拉刷新／驗證分層）・**併發競態三武器＋樂觀鎖** | 共同要求 3、4、7 的完全體 |
| 第 2 節（內幕） | 50 | 記憶體階層・page 與 slotted page・**hexdump 解剖 SQLite 檔**・buffer pool・**現場實作 mini pager** | 把應用寫得更好的「懂」 |
| 課堂實作 | 35 | 把你的專題接上完整 CRUD 分頁＋競態防護測試 | 專題進度：CRUD 能動、防護就位 |

> **本課分水嶺之二**：從今天起每個單元都是「應用半場＋內幕半場」——先把手上的做完，再看引擎裡發生什麼。

# 第 1 節（應用）：把 App 長成完整 CRUD

## 1.1 完整應用的解剖圖

上個單元的福利社會「買」了；一個**完整**的應用還要能：**改**（編輯商品）、**刪**（下架）、
**管**（每張表都有後台）。分層不變：

```
 Gradio UI（薄）── 只負責收值、顯示、刷新
     │  呼叫
 資料層函數（厚）── 驗證 → ? 傳值 → with con: 交易 → 人話訊息
     │  執行
 SQLite ── 約束是最後防線（多人同時操作只有它靠得住 → 1.5 證明）
```

今天照這張圖，把福利社升級成**完整版**——你的專題照同樣的路走一遍就完工一半。

In [ ]:
#@title 📦 重建福利社（U05 的老朋友，加了 active 欄位當「軟刪除」開關）
import sqlite3, os
import numpy as np, pandas as pd

if os.path.exists("shop.db"):
    os.remove("shop.db")
sh = sqlite3.connect("shop.db", check_same_thread=False)
sh.row_factory = sqlite3.Row
sh.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE card(
  card_id INTEGER PRIMARY KEY, owner TEXT NOT NULL,
  balance INTEGER NOT NULL DEFAULT 0 CHECK (balance >= 0));
CREATE TABLE product(
  pid INTEGER PRIMARY KEY, pname TEXT NOT NULL UNIQUE,
  price INTEGER NOT NULL CHECK (price > 0),
  stock INTEGER NOT NULL DEFAULT 0 CHECK (stock >= 0),
  active INTEGER NOT NULL DEFAULT 1 CHECK (active IN (0, 1)));   -- 1=販售中 0=已下架
CREATE TABLE purchase(
  purchase_id INTEGER PRIMARY KEY,
  card_id INTEGER NOT NULL REFERENCES card(card_id),
  at TEXT NOT NULL DEFAULT (datetime('now','localtime')));
CREATE TABLE purchase_item(
  purchase_id INTEGER NOT NULL REFERENCES purchase(purchase_id) ON DELETE CASCADE,
  pid INTEGER NOT NULL REFERENCES product(pid),
  qty INTEGER NOT NULL CHECK (qty > 0),
  unit_price INTEGER NOT NULL,
  PRIMARY KEY (purchase_id, pid));
""")
sh.executemany("INSERT INTO card(owner, balance) VALUES (?,?)",
               [("林佳蓉", 500), ("陳威廷", 120), ("吳孟軒", 60)])
sh.executemany("INSERT INTO product(pname, price, stock) VALUES (?,?,?)",
               [("咖啡", 45, 10), ("餅乾", 25, 8), ("泡麵", 38, 5), ("能量飲", 55, 1)])
sh.commit()
print("shop.db 就緒 ✅（能量飲只剩 1 罐——待會的競態主角）")

## 1.2 表單驗證的三層防線

| 層 | 誰做 | 擋什麼 | 例 |
|---|---|---|---|
| ① UI 元件 | Gradio | 型別與範圍「根本輸不進來」 | `gr.Slider(1, 20)`、Dropdown 只給合法選項 |
| ② 程式驗證 | 資料層函數 | 商業規則、給**人話**訊息 | 名稱去空白、價格上限、跨欄檢查 |
| ③ 約束 | SQLite | **最後防線**（多人同時也擋得住） | UNIQUE／CHECK／FK |

心法：①讓錯誤「難發生」、②讓錯誤「看得懂」、③讓錯誤「進不了資料庫」。三層都要有——只靠一層都會漏。

### 隨堂快答：下面四種錯，各是哪一層的職責？

負庫存？重複劃位？名稱全空白？停卡還想購買？

<details><summary>答案</summary>
負庫存＝③ CHECK（②也該先講人話）；重複劃位＝③ UNIQUE（多人同時只有它擋得住）；名稱空白＝②（①的必填提示輔助）；停卡購買＝②商業規則（庫裡它是合法資料，約束管不到「業務狀態的組合」）。
一句話：**格式歸 ①②、事實歸 ③、業務規則歸 ②＋交易**。
</details>

In [ ]:
# 驗證器 pattern：回傳「錯誤清單」——空清單 = 過關（比丟例外好組合、好測試）
def validate_product(pname, price, stock):
    errs = []
    pname = (pname or "").strip()
    if not pname:                 errs.append("名稱必填")
    if len(pname) > 30:           errs.append("名稱太長（≤30 字）")
    try:
        price = int(price); stock = int(stock)
        if not (1 <= price <= 10_000):  errs.append("單價要在 1–10,000")
        if not (0 <= stock <= 999):     errs.append("庫存要在 0–999")
    except (TypeError, ValueError):
        errs.append("單價與庫存要是整數")
    return errs, pname

print(validate_product("  茶葉蛋 ", 13, 30))       # 過關：([], '茶葉蛋')
print(validate_product("", -5, 10**6))             # 三個錯一次報齊（比一次丟一個例外友善）
assert validate_product("茶葉蛋", 13, 30)[0] == []
assert len(validate_product("", -5, 10**6)[0]) == 3
print("✅ 驗證器：一次把所有錯講完——使用者不用玩「猜哪裡錯」的遊戲")

In [ ]:
# 資料層 v2：商品的 C（新增）與 U（編輯）——編輯是「整列更新」，配 rowcount 檢查
def add_product(pname, price, stock):
    errs, pname = validate_product(pname, price, stock)
    if errs:
        return "⚠️ " + "、".join(errs)
    try:
        with sh:
            sh.execute("INSERT INTO product(pname, price, stock) VALUES (?,?,?)",
                       (pname, int(price), int(stock)))
        return f"✅ 已上架「{pname}」"
    except sqlite3.IntegrityError:
        return f"❌ 已有同名商品「{pname}」"

def edit_product(pid, pname, price, stock):
    errs, pname = validate_product(pname, price, stock)
    if errs:
        return "⚠️ " + "、".join(errs)
    try:
        with sh:
            n = sh.execute("""UPDATE product SET pname = ?, price = ?, stock = ?
                              WHERE pid = ?""", (pname, int(price), int(stock), int(pid))).rowcount
        return "✅ 已更新" if n else "⚠️ 查無此商品（可能剛被別人下架）"
    except sqlite3.IntegrityError:
        return "❌ 名稱與其他商品撞名"

print(add_product("茶葉蛋", 13, 30))
print(add_product("茶葉蛋", 15, 10))          # UNIQUE 擋
print(edit_product(5, "茶葉蛋（特價）", 10, 30))
assert edit_product(99999, "x", 10, 1).startswith("⚠️")
print("✅ C 與 U 完成——注意編輯後 rowcount=0 的訊息：它替你抓到「改到不存在的列」")

## 1.3 刪除的兩種姿勢：硬刪 vs 軟刪

| | 硬刪 `DELETE` | 軟刪 `active = 0` |
|---|---|---|
| 歷史紀錄 | FK 擋你（有明細就刪不掉）——**這是保護不是找碴** | 完整保留 |
| 報表 | 舊訂單 join 不到商品 → 破洞 | 一切照舊 |
| 「復活」 | 不可能 | `active = 1` 一秒 |
| 適用 | 真的建錯、從沒被引用過 | **預設選它**（商品、會員、房型⋯⋯） |

你的專題：**被交易表引用過的主檔，一律軟刪**。demo 兩種都寫給你看：

In [ ]:
def deactivate_product(pid):
    with sh:
        n = sh.execute("UPDATE product SET active = 0 WHERE pid = ? AND active = 1", (pid,)).rowcount
    return "✅ 已下架（歷史訂單完好）" if n else "⚠️ 查無此商品或已下架"

def hard_delete_product(pid):
    try:
        with sh:
            n = sh.execute("DELETE FROM product WHERE pid = ?", (pid,)).rowcount
        return "🗑 已永久刪除" if n else "⚠️ 查無此商品"
    except sqlite3.IntegrityError:
        return "❌ 有歷史訂單引用這個商品——FK 擋下（改用下架）"

def product_table(include_inactive=False):
    sql = ("SELECT pid, pname, price, stock, "
           "CASE active WHEN 1 THEN '販售中' ELSE '已下架' END AS 狀態 FROM product")
    if not include_inactive:
        sql += " WHERE active = 1"
    return pd.read_sql_query(sql + " ORDER BY pid", sh)

# 先製造一筆歷史訂單，讓「硬刪」有東西可撞
with sh:
    sh.execute("INSERT INTO purchase(card_id) VALUES (1)")
    sh.execute("INSERT INTO purchase_item VALUES (1, 1, 1, 45)")
    sh.execute("UPDATE product SET stock = stock - 1 WHERE pid = 1")
    sh.execute("UPDATE card SET balance = balance - 45 WHERE card_id = 1")

print("硬刪 1 號（有訂單）：", hard_delete_product(1))
print("下架 1 號　　　　　：", deactivate_product(1))
full_tbl = product_table(include_inactive=True)
print(full_tbl.to_string(index=False))
assert hard_delete_product(1).startswith("❌")                    # 有歷史 → 硬刪永遠被 FK 擋
assert full_tbl.loc[full_tbl.pid == 1, "狀態"].iloc[0] == "已下架"
print("✅ 心法：FK 報錯不是阻礙，是資料庫在說「這東西有歷史，請軟刪」")

## 1.4 編輯／刪除的 UI 流程：`.select` 三部曲

完整 CRUD 介面的標準流程（U05 學過零件，今天組成整套）：

```
① 點表格一列（.select 事件）→ 把該列的值「帶入表單」＋記住 pid
② 改表單 → 按「儲存」→ edit_product(pid, ...)
③ 或按「下架」→ deactivate_product(pid)
④ 不管哪條路：訊息＋表格＋下拉選單 全部刷新
```

In [ ]:
import sys, importlib.util, subprocess
if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"])
import gradio as gr

def fill_form(evt: gr.SelectData, tbl_df):
    row = tbl_df.iloc[evt.index[0]]
    return (int(row["pid"]), row["pname"], int(row["price"]), int(row["stock"]),
            f"編輯中：#{row['pid']} {row['pname']}")

def ui_save(pid, pname, price, stock):
    return edit_product(pid, pname, price, stock), product_table(True)

def ui_deactivate(pid):
    return deactivate_product(pid), product_table(True)

def ui_add(pname, price, stock):
    return add_product(pname, price, stock), product_table(True)

with gr.Blocks(title="商品管理") as admin_ui:
    gr.Markdown("## ⚙️ 商品管理（點一列 → 帶入表單 → 儲存或下架）")
    tbl = gr.Dataframe(value=product_table(True), interactive=False)
    selected = gr.Number(value=0, visible=False)
    with gr.Row():
        f_name = gr.Textbox(label="名稱"); f_price = gr.Number(label="單價", value=10)
        f_stock = gr.Number(label="庫存", value=0)
    msg = gr.Textbox(interactive=False, label="狀態")
    with gr.Row():
        gr.Button("新增為新商品").click(ui_add, [f_name, f_price, f_stock], [msg, tbl])
        gr.Button("儲存變更", variant="primary").click(ui_save, [selected, f_name, f_price, f_stock], [msg, tbl])
        gr.Button("下架選中", variant="stop").click(ui_deactivate, [selected], [msg, tbl])
    tbl.select(fill_form, [tbl], [selected, f_name, f_price, f_stock, msg])

print("✅ 編輯三部曲建好——「同一個表單」兼做新增與編輯，靠隱藏的 pid 分流")

In [ ]:
# [SKIP-TEST] 管理台試營運
admin_ui.launch(height=560)

## 1.4b 三個後台常備 pattern（大表分頁・批次操作・匯出）

真實後台還有三件天天用的事，各給一個最小可用寫法：

In [ ]:
# ① 大表分頁：別一次全撈——LIMIT ? OFFSET ? ＋ 頁碼（頁碼放 gr.Number 或 gr.State）
PAGE_SIZE = 3
def page_view(page_no):
    off = max(0, int(page_no) - 1) * PAGE_SIZE
    df = pd.read_sql_query("SELECT pid, pname, price, stock FROM product ORDER BY pid LIMIT ? OFFSET ?",
                           sh, params=(PAGE_SIZE, off))
    total = sh.execute("SELECT COUNT(*) FROM product").fetchone()[0]
    return df, f"第 {max(1, int(page_no))} 頁／共 {(total + PAGE_SIZE - 1) // PAGE_SIZE} 頁（{total} 筆）"

for p in (1, 2):
    df, info = page_view(p)
    print(info); print(df.to_string(index=False)); print()
# 介面：上一頁/下一頁按鈕 ±1 頁碼 → page_view → 更新表格。萬列大表也秒開

In [ ]:
# ①b 分頁接上 UI：上一頁／下一頁（page_view 的 Blocks 組裝——後台大表直接抄）
with gr.Blocks() as paging_ui:
    page_state = gr.Number(value=1, visible=False)
    tbl_page = gr.Dataframe(value=page_view(1)[0], interactive=False)
    info_box = gr.Textbox(value=page_view(1)[1], interactive=False, label="頁次")
    def go(page_no, delta):
        page_no = max(1, int(page_no) + delta)
        df, info = page_view(page_no)
        return page_no, df, info
    with gr.Row():
        gr.Button("⬅ 上一頁").click(lambda p: go(p, -1), [page_state], [page_state, tbl_page, info_box])
        gr.Button("下一頁 ➡").click(lambda p: go(p, +1), [page_state], [page_state, tbl_page, info_box])
print("✅ 分頁 UI 組好——頁碼藏在 gr.Number(visible=False)，按鈕只做 ±1")

In [ ]:
# ② 批次操作：先「預覽會動到誰」再執行——U02 那句「改資料前先用同條件 SELECT」的應用版
def restock_preview(threshold=6):
    return pd.read_sql_query(
        "SELECT pid, pname, stock FROM product WHERE active = 1 AND stock < ?", sh, params=(threshold,))

def restock_all(threshold=6, up_to=10):
    with sh:
        n = sh.execute("UPDATE product SET stock = ? WHERE active = 1 AND stock < ?",
                       (up_to, threshold)).rowcount
    return f"✅ 補貨 {n} 項到 {up_to} 件"

print("預覽（低於 6 件的）："); print(restock_preview().to_string(index=False))
msg = restock_all(); print(msg)
assert restock_preview().empty and msg.startswith("✅")
print("✅ 批次三步：預覽 → 執行 → rowcount 對帳（跟預覽筆數對得上才安心）")

In [ ]:
# ③ 報表匯出 CSV（老師/店長永遠會要 Excel）：to_csv 寫出 → 回讀驗證
report_df = pd.read_sql_query("""SELECT p.pname, SUM(i.qty) AS 銷量, SUM(i.qty * i.unit_price) AS 營收
                                FROM purchase_item i JOIN product p ON i.pid = p.pid
                                GROUP BY p.pid ORDER BY 營收 DESC""", sh)
report_df.to_csv("sales_report.csv", index=False)
readback = pd.read_csv("sales_report.csv")
print(readback.to_string(index=False))
assert len(readback) == len(report_df)
print("✅ sales_report.csv 匯出成功（Colab：左側檔案面板下載；介面可加一顆「匯出」按鈕做同一件事）")

## 1.4c 加碼：讓資料庫自己記日誌——trigger 一瞥

「誰在什麼時候把價格改掉了？」——與其相信每段程式都記得寫 log，不如叫**資料庫自己記**：
trigger ＝ 掛在表上的「事件 → 動作」。好用但難 debug，本課點到為止（專題選用：調價日誌、狀態變更日誌都適合）。

In [ ]:
sh.executescript("""
CREATE TABLE IF NOT EXISTS price_log(
  pid INTEGER, old_price INTEGER, new_price INTEGER,
  at TEXT NOT NULL DEFAULT (datetime('now','localtime')));
DROP TRIGGER IF EXISTS trg_price;
CREATE TRIGGER trg_price
AFTER UPDATE OF price ON product
WHEN OLD.price <> NEW.price
BEGIN
  INSERT INTO price_log(pid, old_price, new_price) VALUES (OLD.pid, OLD.price, NEW.price);
END;
""")
print(edit_product(2, "餅乾", 28, 10))               # 改價 → trigger 自動補一筆日誌
print(edit_product(2, "餅乾", 28, 12))               # 只改庫存、價格沒變 → WHEN 條件擋掉，不記
log = pd.read_sql_query("SELECT * FROM price_log", sh)
print(log.to_string(index=False))
assert len(log) == 1
print("✅ 不管誰、用哪段程式改價，日誌都不會漏——因為記日誌的人是資料庫自己")

### 隨堂練習 A2（2 分鐘口頭）：trigger 該用在哪、不該用在哪？

以下哪些適合 trigger？① 調價寫日誌　② 訂單成立時扣庫存　③ 狀態變更記歷程　④ 報名滿額自動開候補

<details><summary>答案</summary>
①③ 適合（**審計／日誌**類：被動記錄、不改業務流）；②④ 不適合——那是**業務邏輯**，藏進 trigger 之後
「程式碼裡看不到它發生」，除錯像抓鬼、測試也難寫。業務流程留在資料層函數（交易包好），trigger 只當書記官。
</details>

### 隨堂練習 A（5 分鐘）：幫卡片也做一套

照 1.2–1.4 的模板，完成卡片的管理：`validate_card(owner, balance)`（名稱必填、餘額 0–10,000）、
`edit_card(card_id, owner)`、以及「停卡」該用硬刪還是軟刪？先想清楚再寫。

<details><summary>參考方向</summary>

停卡要**軟刪**（卡片被 purchase 引用）：`card` 加 `active` 欄（`ALTER TABLE card ADD COLUMN active INTEGER NOT NULL DEFAULT 1`），
停卡＝`active=0`＋購買函數加一條「停卡不能買」檢查。整套函數長得跟商品版幾乎一樣——**CRUD 是有模板的**，
寫熟一張表，其他表都是抄自己。
</details>

In [ ]:
# 練習 A 工作區
# TODO：validate_card(owner, balance) → 錯誤清單；edit_card(card_id, owner)；停卡（軟刪）





## 1.5 併發競態：專題共同要求第 4 條的完全體

上個單元福利社的 `buy()` 裡那句 `UPDATE ... WHERE stock >= ?` 你已經用過了。
今天把「為什麼」跟「還有哪些武器」一次講清楚——**你的專題要現場 demo 這一段**。

### 事故現場：check-then-act

「先 SELECT 檢查庫存，夠了再 UPDATE 扣」——兩個人可以**同時通過檢查**：

In [ ]:
# 兩條連線 = 兩個同時按下「購買」的人；目標：最後 1 罐能量飲
sh.execute("UPDATE product SET stock = 1 WHERE pname = '能量飲'"); sh.commit()
con_a = sqlite3.connect("shop.db"); con_b = sqlite3.connect("shop.db")

a_sees = con_a.execute("SELECT stock FROM product WHERE pname='能量飲'").fetchone()[0]
b_sees = con_b.execute("SELECT stock FROM product WHERE pname='能量飲'").fetchone()[0]
print(f"甲 SELECT：剩 {a_sees} → 判定可買；乙 SELECT：剩 {b_sees} → 也判定可買")

# 兩人先後執行「無條件扣一」（他們都已通過檢查）
con_a.execute("UPDATE product SET stock = stock - 1 WHERE pname='能量飲'"); con_a.commit()
try:
    con_b.execute("UPDATE product SET stock = stock - 1 WHERE pname='能量飲'"); con_b.commit()
    print("乙也扣成功了？！")
except sqlite3.IntegrityError as e:
    print(f"乙撞上 CHECK(stock >= 0) → {e}")
print("目前庫存：", sh.execute("SELECT stock FROM product WHERE pname='能量飲'").fetchone()[0])
con_a.close(); con_b.close()
print()
print("→ 這次是 CHECK 當了最後防線（沒有它就是 -1 超賣）。但乙收到的是一臉例外——")
print("  「被擋下」跟「被好好告知沒搶到」是兩回事。所以要有武器一：")

In [ ]:
# 武器一【條件式 UPDATE】：檢查與動作合成一句，rowcount 告訴你搶到沒——扣數量類的首選
sh.execute("UPDATE product SET stock = 1 WHERE pname = '能量飲'"); sh.commit()
con_a = sqlite3.connect("shop.db"); con_b = sqlite3.connect("shop.db")

grab_sql = "UPDATE product SET stock = stock - 1 WHERE pname = '能量飲' AND stock >= 1"
a_got = con_a.execute(grab_sql).rowcount; con_a.commit()
b_got = con_b.execute(grab_sql).rowcount; con_b.commit()
print(f"甲 rowcount={a_got} → {'🎉 搶到' if a_got else '沒搶到'}")
print(f"乙 rowcount={b_got} → {'🎉 搶到' if b_got else '😢 沒搶到 → 回「已售完」而不是例外'}")
assert (a_got, b_got) == (1, 0)
con_a.close(); con_b.close()
print("→ 適用：扣庫存、扣名額、扣餘額——「數量類」競態。庫存、名額、餘額類題目就是它。")

In [ ]:
# 武器一的反向題：取消訂單「把庫存加回去」——同一招防「重複取消」（狀態機 × 條件式 UPDATE）
sh.executescript("""
DROP TABLE IF EXISTS order_demo;
CREATE TABLE order_demo(oid INTEGER PRIMARY KEY, pid INTEGER, qty INTEGER,
                        status TEXT NOT NULL DEFAULT '成立' CHECK (status IN ('成立','已取消')));
INSERT INTO order_demo(pid, qty) VALUES (2, 3);
"""); sh.commit()

def cancel_order(oid):
    with sh:
        n = sh.execute("UPDATE order_demo SET status = '已取消' WHERE oid = ? AND status = '成立'",
                       (oid,)).rowcount                     # 條件式 UPDATE：只有「成立」的單能取消
        if n == 0:
            return "⚠️ 查無此單或已取消（不會重複退庫存！）"
        sh.execute("UPDATE product SET stock = stock + (SELECT qty FROM order_demo WHERE oid = ?) WHERE pid = 2", (oid,))
    return "✅ 已取消，庫存退回"

s0 = sh.execute("SELECT stock FROM product WHERE pid = 2").fetchone()[0]
print(cancel_order(1)); print(cancel_order(1))              # 第二次取消必須被擋
s1 = sh.execute("SELECT stock FROM product WHERE pid = 2").fetchone()[0]
print(f"庫存 {s0} → {s1}（只退了一次 ✅）")
assert s1 == s0 + 3
print("→ 「重複按取消」跟「兩人同搶」是同一種病：用狀態當條件，一句話裡完成檢查＋轉移。")

In [ ]:
# 武器二【UNIQUE 約束】：搶「唯一位子」類——資料庫保證同一格只進得去一個人
sh.executescript("""
DROP TABLE IF EXISTS seat_demo;
CREATE TABLE seat_demo(showtime INTEGER, seat TEXT, buyer TEXT,
                       UNIQUE (showtime, seat));
"""); sh.commit()
con_a = sqlite3.connect("shop.db"); con_b = sqlite3.connect("shop.db")
def grab_seat(con, who):
    try:
        con.execute("INSERT INTO seat_demo VALUES (1, 'A1', ?)", (who,)); con.commit()
        return f"{who}：🎉 A1 到手"
    except sqlite3.IntegrityError:
        return f"{who}：😢 A1 已被劃走 → 引導選別的位子"
print(grab_seat(con_a, "甲")); print(grab_seat(con_b, "乙"))
assert sh.execute("SELECT COUNT(*) FROM seat_demo").fetchone()[0] == 1
con_a.close(); con_b.close()
print("→ 適用：座位、時段、同人同活動一筆——「唯一格子」競態。")

In [ ]:
# 武器三【BEGIN IMMEDIATE】：多步驟「讀了才能決定怎麼寫」時，先把寫權整段鎖起來
con_a = sqlite3.connect("shop.db"); con_b = sqlite3.connect("shop.db")
con_b.execute("PRAGMA busy_timeout = 300")       # 乙最多等 0.3 秒（預設會等更久）

con_a.execute("BEGIN IMMEDIATE")                 # 甲先聲明「我要寫」→ 拿到寫權
con_a.execute("UPDATE card SET balance = balance - 10 WHERE card_id = 1")
print("甲已開始交易（還沒 commit），此時乙也想開始……")
import time
t0 = time.time()
try:
    con_b.execute("BEGIN IMMEDIATE")
    print("乙竟然也拿到了？！")
except sqlite3.OperationalError as e:
    print(f"乙被擋：{e}（等了 {time.time()-t0:.2f}s 後放棄）")
con_a.commit()                                    # 甲收工，寫權釋出
con_b.execute("BEGIN IMMEDIATE"); con_b.execute("ROLLBACK")
print("甲 commit 之後，乙重試就拿到了 ✅")
con_a.close(); con_b.close()
print()
print("→ 適用：一段交易裡要「先讀再依結果寫好幾步」（結帳金額試算、批次調撥）。")
print("  App 端配 busy_timeout ＋ 重試；SQLite 一次只允許一個寫者，IMMEDIATE 只是把排隊提早到交易開頭。")

In [ ]:
# 武器三的配件：重試迴圈——「鎖住」不是錯誤，是「等一下再來」
import threading

def with_retry(fn, tries=4):
    for i in range(tries):
        try:
            return fn()
        except sqlite3.OperationalError as e:
            if "locked" not in str(e) or i == tries - 1:
                raise
            time.sleep(0.06 * (i + 1))                    # 指數退讓的簡化版
            print(f"  （第 {i+1} 次撞鎖，退讓後重試）")

con_a = sqlite3.connect("shop.db", check_same_thread=False)   # Timer 執行緒要替甲 commit
con_b = sqlite3.connect("shop.db", timeout=0.05)               # 乙把內建等待調到極短——讓「重試」有戲唱
con_a.execute("BEGIN IMMEDIATE"); con_a.execute("UPDATE card SET balance = balance + 1 WHERE card_id = 1")
threading.Timer(0.25, con_a.commit).start()                    # 0.25 秒後甲才收工

def b_write():
    con_b.execute("BEGIN IMMEDIATE")
    con_b.execute("UPDATE card SET balance = balance + 1 WHERE card_id = 2")
    con_b.commit()
    return "乙：✅ 重試後寫入成功"

result = with_retry(b_write)
print(result)
assert result.startswith("乙：✅")
time.sleep(0.05)
con_a.close(); con_b.close()
print("→ 使用者只覺得「多轉了 0.1 秒」，而不是看到一臉 database is locked。")

### 榮譽第四武器【樂觀鎖】：兩個管理員同時「編輯」同一筆

競態不只發生在「搶」——**編輯畫面**也有：甲乙同時打開商品 #2 的表單，甲存、乙再存，
乙就**默默蓋掉**甲的修改（lost update 的 UI 版）。解法：表加一欄 `version`，更新時「帶著你當初讀到的版本號」：

In [ ]:
# 樂觀鎖：UPDATE ... WHERE id = ? AND version = ?——版本不對＝有人先改過＝這次不給存
sh.executescript("""
DROP TABLE IF EXISTS doc_demo;
CREATE TABLE doc_demo(id INTEGER PRIMARY KEY, note TEXT, version INTEGER NOT NULL DEFAULT 0);
INSERT INTO doc_demo(note) VALUES ('原始說明');
"""); sh.commit()

def save_note(note, seen_version):
    with sh:
        n = sh.execute("""UPDATE doc_demo SET note = ?, version = version + 1
                          WHERE id = 1 AND version = ?""", (note, seen_version)).rowcount
    return "✅ 已儲存" if n else "⚠️ 這筆資料剛被別人改過——請重新載入再編輯"

v0 = sh.execute("SELECT version FROM doc_demo WHERE id = 1").fetchone()[0]
print("甲乙同時打開表單（都看到 version", v0, "）")
print("甲存：", save_note("甲的修改", v0))
print("乙存：", save_note("乙的修改", v0))          # 版本已變 → 乙被擋，不會默默蓋掉甲
final = sh.execute("SELECT note, version FROM doc_demo WHERE id = 1").fetchone()
print("最後內容：", dict(final))
assert final["note"] == "甲的修改"
print("→ 「樂觀」＝不先鎖、存的時候才驗——後台編輯、個人資料頁這種低衝突場景的標準解。")
print("  本質仍是武器一（條件式 UPDATE）：把 version 當成「庫存」來檢查。")

### 競態三（＋一）武器速查（你的專題挑對的用，demo 時講得出為什麼）

| 武器 | 一句話 | 適用競態 | 典型場景 |
|---|---|---|---|
| 條件式 UPDATE ＋ rowcount | 檢查與動作合成一句 | **數量**：庫存／名額／餘額 | 售票、進銷存、報名、儲值 |
| UNIQUE（含部分索引） | 同一格只進一人 | **唯一位**：座位／時段／一人一筆 | 訂票、預約、掛號、填答 |
| BEGIN IMMEDIATE ＋ busy_timeout | 整段交易先佔寫權 | **多步讀寫**：試算後才寫 | 任何複雜結帳流程 |
| （樂觀鎖 version 欄） | 存檔時驗「沒人動過」 | **編輯衝突**：lost update | 後台編輯、表單修改 |

武器**都要**搭 `with con:` 交易與約束（最後防線）。U09 會回來拆：鎖到底是怎麼運作的（2PL／WAL／MVCC）。

### 隨堂練習 B（5 分鐘）：幫你的題目選武器

寫下你題目的「必須原子完成」操作（借出？報名？成交？），回答三題：
1. 它是「數量」還是「唯一位」還是「多步讀寫」（還是編輯衝突）？
2. 用哪個武器？防線約束是哪條？
3. 兩條連線的重現劇本怎麼演（demo 稿先打好）？

<details><summary>對照範例（餐廳訂位）</summary>

訂位＝「時段×桌」的唯一位 → 武器二 `UNIQUE(table_id, slot)`；候位遞補是多步讀寫 → 武器三包整段；
桌數統計快照另配對帳查詢。demo 劇本：兩連線同時 INSERT 同桌同時段 → 一成功一改走候位。
</details>

In [ ]:
# 練習 B 工作區：用福利社當你的替身先排練一次（兩連線劇本）
# ① 選資源：能量飲 stock=1（數量）或 seat_demo（唯一位）
# ② 兩條 connection 各跑一次「防護版」操作
# ③ assert：恰好一人成功
# TODO





# 第 2 節（內幕）：資料到底怎麼躺在磁碟上

## 2.1 為什麼要懂儲存？——記憶體階層的殘酷數量級

| 動作 | 典型耗時 | 相對感受（×10⁹） |
|---|---|---|
| L1 快取 | ~1 ns | 心跳一下 |
| RAM 隨機讀 | ~100 ns | 沖一杯咖啡 |
| SSD 隨機讀 4KB | ~100 µs | 睡一晚 |
| 傳統硬碟尋軌 | ~10 ms | 出國玩一個月 |
| （網路一來回） | ~100 ms+ | 一學期 |

兩個結論支配了資料庫的一切設計：
1. **磁碟比記憶體慢幾千～十萬倍** → 能不碰磁碟就不碰（buffer pool，2.4）；
2. **循序讀寫遠快於隨機** → 資料按「頁」成塊搬運（page，2.2）。

下一格在你的機器上實測——數字不同沒關係，**數量級的落差**才是重點。

In [ ]:
# 你機器的體檢：RAM 隨機取值 vs 檔案循序讀 vs 檔案隨機讀（Colab 是 SSD/雲端碟，差距會比機械碟小）
import os, time, random
random.seed(42)

# (1) RAM：list 隨機取值 ×100 萬
data = list(range(2_000_000))
t = time.time()
s = 0
for _ in range(1_000_000):
    s += data[random.randrange(2_000_000)]
ram_ns = (time.time() - t) / 1_000_000 * 1e9

# (2) 磁碟：先造一個 64 MB 檔
with open("bigfile.bin", "wb") as f:
    f.write(os.urandom(64 * 1024 * 1024))

t = time.time()                                   # 循序整檔讀
with open("bigfile.bin", "rb") as f:
    while f.read(1024 * 1024):
        pass
seq_s = time.time() - t

t = time.time()                                   # 隨機 4KB 讀 ×2000
with open("bigfile.bin", "rb") as f:
    for _ in range(2000):
        f.seek(random.randrange(64 * 1024 * 1024 - 4096))
        f.read(4096)
rand_us = (time.time() - t) / 2000 * 1e6

print(f"RAM 隨機取值　 ≈ {ram_ns:8.0f} ns／次")
print(f"檔案循序讀　　 ≈ {64 / seq_s:8.0f} MB/s（整塊搬最划算）")
print(f"檔案隨機讀 4KB ≈ {rand_us:8.1f} µs／次")
print("\n→ 就算在 SSD／快取加持下，「隨機碰磁碟」仍比 RAM 慢幾個數量級；機械碟時代差距是十萬倍。")
print("  資料庫的兩大生存策略：①盡量別碰磁碟（快取） ②要碰就整頁搬（循序）。")

## 2.2 page：資料庫搬運資料的最小單位

磁碟太慢，所以**沒有人一次讀一列**——一律整「頁」搬（SQLite 預設 **4096 bytes**）。

```
 資料庫檔案 = 一疊 page
 ┌────────────┬────────────┬────────────┬──── ⋯
 │ page 1     │ page 2     │ page 3     │
 │（檔頭+目錄）│（某表的節點）│（某表的節點）│
 └────────────┴────────────┴────────────┴──── ⋯
   每頁裡面（slotted page 佈局）：
   ┌──────────────────────────────┐
   │ 頁首（型別、格數、剩餘空間…）    │
   │ 格目錄：指向各筆記錄的位移 ↓     │
   │        （中間是空洞，兩邊長）    │
   │ 記錄 3 ← 記錄 2 ← 記錄 1（從尾端往回長）│
   └──────────────────────────────┘
```

- 一列（record）被編碼成一串 bytes，塞進某頁的某格——**「第幾頁第幾格」就是它的住址**。
- 變長記錄靠「格目錄」間接定位：紀錄搬家只改目錄，住址不變。
- U07 的索引，本質就是「幫住址建捷徑」。

口說無憑——下一格直接打開 SQLite 檔案看。

In [ ]:
#@title 🔬 hexdump 解剖：SQLite 檔案的第一個 100 bytes（規格書等級的檔頭）
import sqlite3, struct

con6 = sqlite3.connect("anatomy.db")
con6.executescript("""
DROP TABLE IF EXISTS student;
CREATE TABLE student(sid TEXT PRIMARY KEY, name TEXT, dept TEXT);
INSERT INTO student VALUES ('S001','林佳蓉','統計'),('S002','陳威廷','統計'),('S007','吳孟軒','資訊');
""")
con6.commit(); con6.close()

raw = open("anatomy.db", "rb").read()
head = raw[:100]
print("前 32 bytes 的 hexdump：")
for i in range(0, 32, 16):
    hexs = " ".join(f"{b:02x}" for b in head[i:i+16])
    text = "".join(chr(b) if 32 <= b < 127 else "." for b in head[i:i+16])
    print(f"  {i:04x}  {hexs}  |{text}|")

print("\n照規格書拆欄位（https://sqlite.org/fileformat2.html）：")
print("  offset 0–15  魔術字串 →", raw[:16])
print("  offset 16–17 page 大小 →", struct.unpack('>H', raw[16:18])[0], "bytes")
print("  offset 28–31 總頁數　 →", struct.unpack('>I', raw[28:32])[0], "頁")
print("  offset 44–47 schema 版本（改一次 schema 加一）→", struct.unpack('>I', raw[44:48])[0])
print("\n→ 檔案大小 =", len(raw), "bytes = 頁數 × 頁大小 ✅（資料庫檔＝一疊 page，句點）")

In [ ]:
# 你的資料真的躺在某一頁裡：把一筆認得出來的記錄塞進去，然後用 bytes 搜出它的住址
con6 = sqlite3.connect("anatomy.db")
con6.execute("INSERT INTO student VALUES ('S999', 'ZZ找我ZZ', '統計')")
con6.commit(); con6.close()

raw = open("anatomy.db", "rb").read()
page_size = struct.unpack('>H', raw[16:18])[0]
marker = "ZZ找我ZZ".encode("utf-8")
pos = raw.find(marker)
print(f"'ZZ找我ZZ' 的 UTF-8 bytes 出現在檔案 offset {pos:,}")
print(f"→ 它住在第 {pos // page_size + 1} 頁（offset ÷ {page_size}，從 1 數起）、頁內第 {pos % page_size} byte")
assert pos > 0
print("\n附近的原始 bytes（記錄是「型別串＋值串」的緊湊編碼，中文就是 UTF-8）：")
start = max(0, pos - 16)
print(" ", " ".join(f"{b:02x}" for b in raw[start:pos+len(marker)+4]))

In [ ]:
# 隨堂練習 D 工作區：換你當法醫——插入一筆「你的名字」，算出它住第幾頁第幾 byte
# ① con6 重開 anatomy.db，INSERT 一筆含獨特字串的記錄，commit
# ② 讀檔 → find() → 除以 page_size
# TODO





In [ ]:
# sqlite_master 就在第 1 頁：連 schema 本身都是一筆筆「記錄」（U02 的伏筆回收）
raw = open("anatomy.db", "rb").read()
page1 = raw[:4096]
print("在第 1 頁裡搜 'CREATE TABLE' →", b"CREATE TABLE" in page1)
print("搜表名 student 的 bytes     →", b"student" in page1)
print()
con6 = sqlite3.connect("anatomy.db")
try:
    print("dbstat 虛擬表：每個物件佔了哪些頁——")
    print(pd.read_sql_query("""SELECT name, COUNT(*) AS 頁數, SUM(ncell) AS 記錄數,
                                     SUM(payload) AS 資料bytes
                              FROM dbstat GROUP BY name""", con6).to_string(index=False))
except sqlite3.OperationalError:
    print("（這顆 SQLite 沒編譯 dbstat，用 PRAGMA page_count 也能看總頁數）")
print("\nPRAGMA page_size =", con6.execute("PRAGMA page_size").fetchone()[0],
      "；page_count =", con6.execute("PRAGMA page_count").fetchone()[0])
con6.close()

In [ ]:
# 順手學會資料庫的「健檢指令」：integrity_check——檔案傳來傳去、雲端同步後，先體檢再用
con6 = sqlite3.connect("anatomy.db")
print("PRAGMA integrity_check →", con6.execute("PRAGMA integrity_check").fetchone()[0])
print("PRAGMA quick_check     →", con6.execute("PRAGMA quick_check").fetchone()[0])
con6.close()
print("→ 回 'ok' 表示每一頁、每個索引、每條約束都自洽。專題資料庫從雲端硬碟載回來時，先跑這句。")
print("  （壞檔的常見來源：同步軟體在寫入途中複製檔案——所以 -journal/-wal 檔要跟主檔一起搬！）")

In [ ]:
# 型別與長度是「物理」問題：同樣 5,000 列，欄位胖瘦直接決定佔幾頁
scon = sqlite3.connect("size_demo.db")
scon.executescript("""
DROP TABLE IF EXISTS thin_t; DROP TABLE IF EXISTS fat_t;
CREATE TABLE thin_t(id INTEGER PRIMARY KEY, g INTEGER);
CREATE TABLE fat_t(id INTEGER PRIMARY KEY, g INTEGER, memo TEXT);
""")
with scon:
    scon.executemany("INSERT INTO thin_t VALUES (?,?)", [(i, i % 100) for i in range(5000)])
    scon.executemany("INSERT INTO fat_t VALUES (?,?,?)",
                     [(i, i % 100, "這位同學的修課心得" * 12) for i in range(5000)])
for t in ("thin_t", "fat_t"):
    try:
        n_pages = scon.execute("SELECT COUNT(*) FROM dbstat WHERE name = ?", (t,)).fetchone()[0]
        print(f"{t}：{n_pages:4d} 頁（每頁塞 {5000 // n_pages} 列左右）")
    except sqlite3.OperationalError:
        print(f"{t}：（無 dbstat，略）")
scon.close()
print("→ 一頁塞越多列，同一次 I/O 撈到越多資料——「欄位別亂胖」不只是潔癖，是效能（U07 再見它一次）。")

In [ ]:
# 經典疑問：「我 DELETE 了一半資料，檔案怎麼沒變小？」——freelist 與 VACUUM
scon = sqlite3.connect("size_demo.db")
before = scon.execute("PRAGMA page_count").fetchone()[0]
with scon:
    scon.execute("DELETE FROM fat_t WHERE id % 2 = 0")          # 刪掉一半
after = scon.execute("PRAGMA page_count").fetchone()[0]
free_pages = scon.execute("PRAGMA freelist_count").fetchone()[0]
print(f"刪除前 {before} 頁 → 刪除後 {after} 頁（沒縮！），其中 {free_pages} 頁進了 freelist（待回收再利用）")
scon.execute("VACUUM")                                          # 整檔重建，把空頁真的還給作業系統
print(f"VACUUM 後：{scon.execute('PRAGMA page_count').fetchone()[0]} 頁、"
      f"freelist {scon.execute('PRAGMA freelist_count').fetchone()[0]} 頁")
scon.close()
print("→ DELETE 只是把頁標成「空」，留著給之後的 INSERT 用；要還地就 VACUUM（會鎖檔重寫，離峰做）。")

In [ ]:
# 預告 U09：WAL 模式——旁邊會多出一個 -wal 檔（那就是「先寫日誌」的日誌本體）
for f in ("wal_demo.db", "wal_demo.db-wal", "wal_demo.db-shm"):
    if os.path.exists(f):
        os.remove(f)                                  # 重跑防呆：從乾淨狀態開始
wcon = sqlite3.connect("wal_demo.db")
print("journal_mode →", wcon.execute("PRAGMA journal_mode=WAL").fetchone()[0])
with wcon:
    wcon.execute("CREATE TABLE t(x)"); wcon.execute("INSERT INTO t VALUES (42)")
print("目前資料夾裡：", sorted(f for f in os.listdir(".") if f.startswith("wal_demo")))
wcon.close()
print("→ 改動先寫進 wal_demo.db-wal，之後才「歸檔」回主檔——為什麼這樣就能斷電不掉資料？U09 拆解。")

### 隨堂快答：下面哪些動作「一定」碰磁碟？

① `SELECT`（資料已在快取）　② `INSERT` 之後、`commit` 之前　③ `commit`　④ 重開 Colab 後的第一句查詢

<details><summary>答案</summary>
③④ 必碰：commit 要保證「斷電也還在」，非落地不可（U09 講它怎麼落地才安全）；重啟後快取全空（冷啟動），第一批查詢全是 miss。
①② 通常不碰：讀命中快取；未 commit 的修改先待在記憶體頁（延遲寫回）。——所以「批次匯入慢」的兇手常是「每筆都 commit」（U05 實測過 40 倍差）。
</details>

## 2.3 buffer pool：資料庫的「頁快取」

每次都去磁碟搬頁？慢死。所以引擎在記憶體養一個 **buffer pool**：

```
 查詢要第 7 頁 ──► 在 pool 裡嗎？ ──有──► 直接用（cache hit，奈秒級）
                        │沒有
                        ▼
                  從磁碟搬進來（miss，微秒～毫秒級）
                  pool 滿了 → 踢掉「最久沒用的」（LRU）
```

SQLite 的 pool 大小就是 `PRAGMA cache_size`（預設約 2MB）。實驗看看它的效果：

In [ ]:
# cache_size 實驗：同一批隨機點查，pool 只有 2 頁 vs 2000 頁
import random
bcon = sqlite3.connect("anatomy.db")
bcon.execute("DROP TABLE IF EXISTS big")
bcon.execute("CREATE TABLE big(id INTEGER PRIMARY KEY, txt TEXT)")
with bcon:
    bcon.executemany("INSERT INTO big VALUES (?,?)", [(i, "資料塊" * 20) for i in range(30_000)])

def point_query(n=20_000):
    random.seed(1)
    t = time.time()
    for _ in range(n):
        bcon.execute("SELECT txt FROM big WHERE id = ?", (random.randrange(30_000),)).fetchone()
    return time.time() - t

bcon.execute("PRAGMA cache_size = 2");    t_small = point_query()
bcon.execute("PRAGMA cache_size = 2000"); t_big = point_query()
print(f"pool =    2 頁：{t_small*1000:6.0f} ms")
print(f"pool = 2000 頁：{t_big*1000:6.0f} ms   → 快 {t_small/t_big:.2f} 倍")
bcon.close()
print()
print("→ 差距怎麼「才」這樣？因為下面還墊著一層 OS 的檔案快取幫你擋（快取是階層！）。")
print("  真正的磁碟 miss 在雲端硬碟/機械碟上是毫秒級——那時 pool 就是生死線。")
print("  你能帶走的：資料庫的『快』大半是快取的功勞；冷啟動第一批查詢慢，是 miss 不是壞掉。")

## 2.4 教師現場實作：mini pager（約 40 行）

規格——工業引擎的縮小版，但機制一比一：
1. 檔案切成 **4KB 頁**，`get(頁號)` 讀、`put(頁號, bytes)` 寫；
2. 記憶體只留 `cap` 頁（**LRU**：滿了踢最久沒用的）；
3. 髒頁（改過的）被踢出或關檔時才寫回磁碟；
4. **記帳**：邏輯讀（上層要了幾次）vs 磁碟讀寫（真的碰了幾次）——快取的成績單。

In [ ]:
from collections import OrderedDict

class MiniPager:
    PAGE = 4096

    def __init__(self, path, cap=8):
        self.f = open(path, "w+b")
        self.cap = cap
        self.cache = OrderedDict()                      # 頁號 -> bytearray（保序＝LRU 隊伍）
        self.dirty = set()
        self.stats = dict(logical=0, hit=0, disk_read=0, disk_write=0)

    def get(self, no):
        self.stats["logical"] += 1
        if no in self.cache:                            # ── cache hit
            self.stats["hit"] += 1
            self.cache.move_to_end(no)                  # 剛用過 → 搬到隊伍尾（最不該被踢）
            return self.cache[no]
        self.f.seek(no * self.PAGE)                     # ── miss：真的去磁碟
        page = bytearray(self.f.read(self.PAGE).ljust(self.PAGE, b"\x00"))
        self.stats["disk_read"] += 1
        self.cache[no] = page
        if len(self.cache) > self.cap:                  # 滿了 → 踢隊伍頭（最久沒用）
            old_no, old_page = self.cache.popitem(last=False)
            if old_no in self.dirty:
                self._write_back(old_no, old_page)
        return page

    def put(self, no, data, off=0):
        page = self.get(no)
        page[off:off + len(data)] = data
        self.dirty.add(no)                              # 只改記憶體，先不碰磁碟！

    def _write_back(self, no, page):
        self.f.seek(no * self.PAGE); self.f.write(page)
        self.stats["disk_write"] += 1
        self.dirty.discard(no)

    def close(self):
        for no in list(self.dirty):                     # 關檔前把髒頁全部落地
            self._write_back(no, self.cache[no])
        self.f.close()

print("MiniPager 就緒——四個機制：整頁搬運／LRU 快取／延遲寫回／IO 記帳")

In [ ]:
# 實驗 1：寫 200 頁再「循序讀兩輪」——第二輪還是得碰磁碟（cap=8 裝不下 200 頁）
pg = MiniPager("mini.db", cap=8)
for no in range(200):
    pg.put(no, f"page-{no:03d} 的內容".encode())
for rnd in (1, 2):
    for no in range(200):
        pg.get(no)
print("循序讀兩輪 →", pg.stats)
seq_hit = pg.stats["hit"] / pg.stats["logical"]
print(f"命中率 {seq_hit:.1%}——LRU 對「掃過就不回頭」的循序讀幫不上忙（剛踢掉的馬上又要）")
pg.close()

In [ ]:
# 實驗 2：同樣 200 頁，但存取像真實系統——80% 的請求集中在 20% 的熱門頁（長尾又來了）
pg = MiniPager("mini.db", cap=8)
random.seed(7)
for _ in range(4000):
    if random.random() < 0.8:
        no = random.randrange(20)                       # 熱門 20 頁
    else:
        no = random.randrange(200)
    pg.get(no)
hot_hit = pg.stats["hit"] / pg.stats["logical"]
print("長尾存取 →", pg.stats)
print(f"命中率 {hot_hit:.1%}——只有 8 頁的快取就接住了大多數請求！")
assert hot_hit > seq_hit
pg.close()
print()
print("→ 快取有效的前提是「存取有熱點」；幸好真實世界幾乎都有（U03 的 Zipf 長尾）。")
print("  SQLite 的 pager ＝ 這個玩具的工業版（加上鎖、journal、WAL——U09 補完）。")

In [ ]:
# 實驗 3：延遲寫回到底賺了什麼——同一頁改 100 次，磁碟只寫 1 次（合併寫入）
pg = MiniPager("mini.db", cap=8)
for i in range(100):
    pg.put(0, f"第 {i} 版".encode())                    # 100 次修改全打在第 0 頁
print("改完 100 次，寫回磁碟了嗎？ disk_write =", pg.stats["disk_write"], "（還沒！都在記憶體）")
pg.close()                                              # 關檔才落地
print("close() 之後（重新統計最後狀態）：髒頁在關檔時一次寫回 → 磁碟只挨了 1 次寫")
pg2 = MiniPager("mini_check.db", cap=8)
for i in range(100):
    pg2.put(0, b"x")
w_before = pg2.stats["disk_write"]
pg2.close()
print(f"驗證：100 次 put 期間 disk_write = {w_before}；這就是「延遲寫回」賺到的合併寫入。")
assert w_before == 0
print("→ 賺：百改一寫。賭：落地前斷電，100 次修改全蒸發——這筆帳 U09 的 journal/WAL 來還。")

In [ ]:
# 實驗 4：把 cap 從 2 調到 64，畫「快取大小 vs 命中率」——你人生第一條快取曲線
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

caps, hits = [], []
for cap in (2, 4, 8, 16, 32, 64):
    pg = MiniPager("mini.db", cap=cap)
    random.seed(7)
    for _ in range(4000):
        pg.get(random.randrange(20) if random.random() < 0.8 else random.randrange(200))
    caps.append(cap); hits.append(pg.stats["hit"] / pg.stats["logical"])
    pg.close()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(caps, [h * 100 for h in hits], marker="o")
ax.set_xscale("log", base=2)
ax.set_xlabel("cache size (pages)"); ax.set_ylabel("hit rate (%)")
ax.set_title("Cache size vs hit rate (80/20 access)")
plt.tight_layout(); plt.show()
for c, h in zip(caps, hits):
    print(f"cap={c:3d} → 命中率 {h:.1%}")
print("→ 曲線先陡後平：cache 蓋過「熱點」之後就邊際效益遞減——PRAGMA cache_size 的調法同理。")

## 2.5 在 pager 上蓋一層「表」：mini 記錄層

pager 只認「頁」；表要的是「第 n 筆記錄」。中間的翻譯層：**記錄住址 = (頁號, 格號)**。
教學版用固定 64-byte 格（真品是變長記錄＋格目錄，也就是 2.2 的 slotted page；機制相同、簿記更多）。

In [ ]:
class MiniTable:
    SLOT, PER_PAGE = 64, 63                      # 每頁 4096B：留 64B 當頁首，其餘 63 格

    def __init__(self, pager):
        self.pg, self.n = pager, 0

    def insert(self, text):
        rid = self.n
        page_no, slot = divmod(rid, self.PER_PAGE)
        raw = text.encode("utf-8")[:self.SLOT].ljust(self.SLOT, b" ")
        self.pg.put(page_no, raw, off=64 + slot * self.SLOT)
        self.n += 1
        return page_no, slot                      # ← 這筆記錄的「住址」

    def get(self, page_no, slot):
        page = self.pg.get(page_no)
        start = 64 + slot * self.SLOT
        return bytes(page[start:start + self.SLOT]).rstrip(b" ").decode("utf-8")

pg = MiniPager("table.db", cap=4)
t = MiniTable(pg)
addrs = [t.insert(f"第 {i} 筆：統計系的資料庫筆記") for i in range(300)]
print("第 0 筆住在", addrs[0], "・第 299 筆住在", addrs[-1])
print("讀回第 150 筆 →", t.get(*addrs[150]))
print("pager 成績單 →", pg.stats, "（300 筆記錄只碰了幾次磁碟——整頁搬運的威力）")
assert t.get(*addrs[0]).startswith("第 0 筆")
pg.close()

### MiniPager ↔ SQLite 對照表（今天蓋的玩具 vs 工業品）

| 玩具版 | SQLite 真品 | 差在哪 |
|---|---|---|
| `get/put` 4KB 頁 | pager 模組 | 真品還管鎖與 journal |
| OrderedDict LRU | pcache（預設 LRU） | 真品可插拔、算記憶體壓力 |
| 髒頁延遲寫回 | 同 | 真品**先寫日誌再寫頁**（U09 的保命關鍵） |
| 固定 64B 格 | 變長記錄＋格目錄（slotted） | 真品一頁是 B-tree 節點（U07 主角） |
| `stats` 記帳 | `PRAGMA cache_size`／`sqlite3_status` | 觀念相同 |

### 隨堂練習 C（3 分鐘）

1. 實驗 1 的循序掃描，把 `cap` 調到 200 會怎樣？先預測再試。
2. 為什麼 `put()` 不立刻寫磁碟？延遲寫回賺到什麼、賭上什麼？（提示：實驗 3 已經演給你看了）

<details><summary>參考答案</summary>

1. cap ≥ 200 → 第二輪全 hit，命中率 50%（第一輪必 miss）。快取要「裝得下工作集」才有戲。
2. 賺：同一頁改一百次只寫一次磁碟（實驗 3 的合併寫入）。賭：機器在寫回前斷電，改動蒸發——
   所以真的資料庫在改頁之前**先寫日誌**（journal/WAL），這正是 U09「當機不掉資料」的答案。
</details>

In [ ]:
# 練習 C 工作區：把實驗 1 的 cap 調到 200 驗證你的預測；有餘力畫出「循序 vs 長尾」兩條快取曲線
# TODO





# 課堂實作（35 分鐘）：你的專題完成「CRUD＋競態防護」

| # | 任務 | 驗收 |
|---|---|---|
| 1 |（20 分）把主要實體的**編輯／刪除**接上介面（1.4 三部曲；被引用的主檔用軟刪） | 點一列→改→存，表格刷新 |
| 2 |（10 分）核心操作套上**對的武器**（1.5 速查表），寫兩連線重現＋防護測試 | assert 一人成功一人被好好拒絕 |
| 3 |（5 分）跑一次你的「應該失敗」測試組 | 全數 ✅ 擋下 |

做完 1＋2，你的專題已滿足共同要求 3、4、7 的主體——接下來只剩報表湊滿與索引對照（U07）。

### CRUD 完成度自驗表（做完打勾）

- [ ] 新增：驗證器一次報齊所有錯、UNIQUE 撞名有人話訊息
- [ ] 查詢：關鍵字＋至少一個過濾條件、空結果不報錯
- [ ] 編輯：點列帶入表單、rowcount=0 有提示、（加分）樂觀鎖 version
- [ ] 刪除：被引用的主檔軟刪、未引用的可硬刪、FK 錯誤翻成人話
- [ ] 每個函數 ≥1 個 assert；「應該失敗」的至少 2 個

In [ ]:
# 課堂實作工作區 1：編輯／刪除三部曲（模板照 1.4，換成你的表）
mycon = sqlite3.connect("myapp.db", check_same_thread=False)
mycon.row_factory = sqlite3.Row
mycon.execute("PRAGMA foreign_keys = ON")

# TODO：你的 DDL＋種子資料
# TODO：def edit_x(...)／def deactivate_x(...)＋rowcount 檢查
# TODO：Blocks：Dataframe.select 帶入表單 → 儲存／下架 → 刷新
print("工作區 1 就緒")

In [ ]:
# 課堂實作工作區 2：競態重現＋防護（模板照 1.5；把表名欄名換成你的）
# 劇本：
#   A = sqlite3.connect("myapp.db"); B = sqlite3.connect("myapp.db")
#   ① 把某資源設成「只剩最後一個」
#   ② A、B 各自執行你的防護版操作（條件式 UPDATE 或 UNIQUE INSERT）
#   ③ assert：恰好一人成功；資源不為負／不重複；輸家拿到人話訊息
print("工作區 2 就緒——寫完這格，你專題的 demo 亮點就有了")

## 專題進度建議（非繳交）

對照 [syllabus](https://github.com/chang-ye-tu/db/blob/master/syllabus.md) 進度表：**到 U06，第一個 CRUD 分頁能動、競態防護就位**。

1. 主要實體的新增／查詢／**編輯／刪除**全部接上介面（軟刪選對）；
2. 核心業務操作：交易＋對的武器＋**兩連線測試**（這就是報告的 demo 亮點）；
3. 測試累積到 ≥8 個 assert（把今天的競態測試算進去）；
4. 下個單元帶著你的 `project.db` 來——U07 現場對它做索引與效能對照（共同要求 6 當堂完成）。

# 本單元你應該帶走

1. 完整 CRUD 模板：驗證器（錯誤清單）→ 資料層（? 傳值＋交易＋rowcount）→ UI 三部曲（select 帶入表單）；被引用的主檔**軟刪**；trigger 當書記官不當業務員。
2. **競態三＋一武器**：條件式 UPDATE（數量／重複取消）、UNIQUE（唯一位）、BEGIN IMMEDIATE（多步讀寫）、樂觀鎖 version（編輯衝突）——搭 `with con:` 與約束最後防線。
3. 磁碟比 RAM 慢幾個數量級、循序勝隨機 → 資料庫**整頁搬運**（4KB page）＋ **buffer pool（LRU）** 拚命不碰磁碟。
4. SQLite 檔案 = 有規格書的一疊 page：魔術字串、頁大小在檔頭；你的每一筆資料都躺在「第幾頁第幾 byte」；`integrity_check` 是體檢指令。
5. mini pager 四機制：整頁搬運／LRU／延遲寫回（實驗 3：百改一寫）／IO 記帳——快取曲線先陡後平，蓋過熱點就夠。
6. 延遲寫回賭的是當機——journal/WAL 怎麼保命，U09 拆解。

**下個單元**：索引與效能——B+ tree 圖解、`EXPLAIN QUERY PLAN`、複合索引與最左前綴，**並現場對你的專題資料做索引前後對照**（共同要求 6）。讀物：Silberschatz ch14；Ullman ch14；Winand《SQL Performance Explained》。

---
## 附錄 A：本單元 cheatsheet

```python
# ── 完整 CRUD 模板 ──
def validate_x(...):  return errs, cleaned            # 空清單 = 過關
def edit_x(id, ...):
    with con:
        n = con.execute("UPDATE x SET ... WHERE id = ?", (...)).rowcount
    return "✅" if n else "⚠️ 查無此筆"
UPDATE x SET active = 0 WHERE id = ?                 # 軟刪（被引用的主檔一律用）

# ── UI 三部曲 ──
tbl.select(fill_form, [tbl], [hidden_id, form_fields..., msg])   # fn(evt: gr.SelectData, tbl)
save_btn.click(edit_x, [hidden_id, form_fields...], [msg, tbl])

# ── 競態武器 ──
UPDATE t SET n = n - 1 WHERE id = ? AND n >= 1       # ① rowcount==1 才算搶到（數量）
INSERT INTO t(a, b) VALUES (?, ?)                    # ② 靠 UNIQUE(a,b)；IntegrityError = 沒搶到
con.execute("BEGIN IMMEDIATE")                       # ③ 先佔寫權；配 PRAGMA busy_timeout
UPDATE t SET ..., version = version + 1
  WHERE id = ? AND version = ?                       # ④ 樂觀鎖：rowcount=0 → 請重新載入
```

```sql
-- ── 儲存引擎觀測 ──
PRAGMA page_size;  PRAGMA page_count;  PRAGMA freelist_count;
PRAGMA cache_size = 2000;  PRAGMA integrity_check;
SELECT name, COUNT(*) FROM dbstat GROUP BY name;     -- 每個物件佔幾頁（有編譯才有）
VACUUM;                                              -- 空頁還給 OS（離峰做）
```

```python
# 檔頭自己讀（fileformat2.html）
h = open("x.db","rb").read(100)
h[:16]                                   # b'SQLite format 3\x00'
struct.unpack('>H', h[16:18])[0]         # page size
struct.unpack('>I', h[28:32])[0]         # page count
```

## 附錄 B：讀物地圖（本單元）

| 講義小節 | Silberschatz 7e | Garcia-Molina/Ullman/Widom 2e |
|---|---|---|
| 記憶體階層與磁碟 | §12.1–12.4 | §13.2–13.3 |
| page／record／slotted page | §13.1–13.2 | §13.5–13.7 |
| buffer pool | §13.5 | §13.4 |
| SQLite 檔案格式 | —— | https://sqlite.org/fileformat2.html（§1–§2 就夠） |
| 競態與鎖（預告） | §17–§18（U09 正式讀） | §18 |